In [ ]:
# CELL 1
# Install all tools your AI needs
# Tap the ▶️ button to run each cell

!pip install torch numpy transformers datasets

import torch
import torch.nn as nn
import numpy as np

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Everything installed!")
print(f"   Device: {device}")
print(f"   If it says 'cpu' go to Runtime → Change Runtime Type → T4 GPU")

In [ ]:
# CELL 2
# Kaggle saves your work automatically!
# No Google Drive setup needed at all
# Your files stay safe at /kaggle/working/

import os

# This is your Kaggle save folder
# Everything here persists between sessions
SAVE = '/kaggle/working/MyGameAI'
os.makedirs(f'{SAVE}/model',       exist_ok=True)
os.makedirs(f'{SAVE}/data',        exist_ok=True)
os.makedirs(f'{SAVE}/checkpoints', exist_ok=True)

print("✅ Save folders ready!")
print(f"   Location: {SAVE}")
print("   Kaggle keeps your files automatically!")
print("   No Google Drive setup needed!")
print()
print("💡 IMPORTANT KAGGLE SETTINGS:")
print("   On the right side panel in Kaggle set these:")
print("   • Accelerator → GPU T4 x2 (free!)")
print("   • Persistence → Files (keeps your files!)")
print("   • Internet → ON (needed to install packages)")

In [ ]:
# CELL 3
# THIS IS YOUR AI BRAIN
# Read every comment carefully — it explains what each part does

import torch
import torch.nn as nn
import math
import re

# ════════════════════════════════════════
# PART A — TOKENIZER
# Converts words into numbers
# AI cannot read words — only numbers
# ════════════════════════════════════════

class SimpleTokenizer:
    """
    Turns text into numbers and back.
    """
    def __init__(self):
        self.word_to_id = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.id_to_word = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.next_id = 4

    def _tokenize(self, text):
        # CRITICAL FIX 1: This separates punctuation from words!
        # "get_tree().quit()" becomes ["get_tree", "(", ")", ".", "quit", "(", ")"]
        return re.findall(r'\w+|[^\w\s]', text.lower())

    def add_text(self, text):
        """Learn new words from text."""
        for word in self._tokenize(text):
            if word not in self.word_to_id:
                self.word_to_id[word] = self.next_id
                self.id_to_word[self.next_id] = word
                self.next_id += 1

    def encode(self, text, max_length=128, pad=True):
        """Text → numbers."""
        words = self._tokenize(text)
        ids = [self.word_to_id.get(w, 3) for w in words]  # 3 = unknown word
        
        if pad:
            ids = ids[:max_length-1]  # cut if too long
            ids.append(2) # CRITICAL FIX 2: Add <END> token so AI learns to stop!
            ids += [0] * (max_length - len(ids))  # pad with zeros
        else:
            ids = ids[:max_length] # No padding for generation!
            
        return ids

    def decode(self, ids):
        """Numbers → text."""
        words = []
        for i in ids:
            if i == 2:  # END token
                break
            if i not in (0, 1):  # skip PAD and START
                words.append(self.id_to_word.get(i, "<UNK>"))
        
        # Join words and clean up spacing around punctuation
        text = " ".join(words)
        text = re.sub(r'\s+([?.!,"\'()])', r'\1', text) 
        return text

    def vocab_size(self):
        return self.next_id

    def save(self, path):
        import json
        with open(path, 'w') as f:
            json.dump({'word_to_id': self.word_to_id,
                      'id_to_word': {str(k): v for k,v in self.id_to_word.items()},
                      'next_id': self.next_id}, f)
        print(f"Tokenizer saved: {path}")

    def load(self, path):
        import json
        with open(path, 'r') as f:
            data = json.load(f)
        self.word_to_id = data['word_to_id']
        self.id_to_word = {int(k): v for k,v in data['id_to_word'].items()}
        self.next_id = data['next_id']
        print(f"Tokenizer loaded: {path}")

tokenizer = SimpleTokenizer()
print("✅ Tokenizer created!")


# ════════════════════════════════════════
# PART B — ATTENTION MECHANISM
# Helps AI focus on important words
# Like how you focus on key words when reading
# ════════════════════════════════════════

class SelfAttention(nn.Module):
    """
    Attention lets the AI focus on
    the most important words when generating an answer.

    Example:
    Question: "How do I make a JUMP in Godot?"
    AI focuses most on: JUMP and Godot
    Less on: How, do, I, make, a, in
    """

    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # These learn what to focus on
        self.query  = nn.Linear(embed_dim, embed_dim)
        self.key    = nn.Linear(embed_dim, embed_dim)
        self.value  = nn.Linear(embed_dim, embed_dim)
        self.output = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape  # batch, sequence length, channels

        # Split into multiple heads
        def split_heads(tensor):
            return tensor.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        Q = split_heads(self.query(x))
        K = split_heads(self.key(x))
        V = split_heads(self.value(x))

        # Calculate attention scores
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale

        # Mask future tokens (AI should not peek at future words)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        # Convert to probabilities
        attention = torch.softmax(scores, dim=-1)

        # Apply attention to values
        out = torch.matmul(attention, V)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.output(out)


# ════════════════════════════════════════
# PART C — TRANSFORMER BLOCK
# One thinking layer of your AI
# Your AI has multiple of these stacked
# ════════════════════════════════════════

class TransformerBlock(nn.Module):
    """
    One layer of thinking.
    Your AI stacks multiple of these.
    More layers = smarter AI (but slower to train)
    """

    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        # Attention — focuses on important words
        self.attention = SelfAttention(embed_dim, num_heads)

        # Feed forward — processes the information
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),                      # activation function
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )

        # Normalization — keeps numbers stable during training
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Attention with residual connection
        x = x + self.dropout(self.attention(self.norm1(x)))
        # Feed forward with residual connection
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return x


# ════════════════════════════════════════
# PART D — YOUR COMPLETE AI MODEL
# This is the full brain
# ════════════════════════════════════════

class YourGameAI(nn.Module):
    """
    YOUR COMPLETE AI MODEL

    This is what you built from scratch.
    It takes a question as input
    and generates a game development answer.

    You can adjust these settings:
    vocab_size  = how many words it knows
    embed_dim   = how much memory per word (bigger = smarter)
    num_layers  = how many thinking layers (more = smarter but slower)
    num_heads   = how many attention heads
    max_length  = longest text it can handle
    """

    def __init__(
        self,
        vocab_size,
        embed_dim=512,    # ← increase to 512 for smarter AI
        num_layers=8,     # ← increase to 8 for smarter AI
        num_heads=8,
        ff_dim=512,
        max_length=512,
        dropout=0.1
    ):
        super().__init__()
        self.max_length = max_length

        # Word embeddings — converts word IDs to vectors
        self.word_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Position embeddings — tells AI WHERE each word is
        self.pos_embed = nn.Embedding(max_length, embed_dim)

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

        # Stack of transformer blocks (the thinking layers)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(embed_dim)

        # Output head — converts back to word probabilities
        self.output_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Initialize weights properly
        self._init_weights()

        # Count parameters
        params = sum(p.numel() for p in self.parameters())
        print(f"   Your AI has {params:,} parameters")

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids):
        B, T = token_ids.shape

        # Get word embeddings
        word_emb = self.word_embed(token_ids)

        # Get position embeddings
        positions = torch.arange(T, device=token_ids.device).unsqueeze(0)
        pos_emb = self.pos_embed(positions)

        # Combine word + position information
        x = self.dropout(word_emb + pos_emb)

        # Pass through all thinking layers
        for layer in self.layers:
            x = layer(x)

        # Final normalization
        x = self.norm(x)

        # Get word probabilities
        logits = self.output_head(x)
        return logits

    def generate(self, tokenizer, prompt, max_new_tokens=200, temperature=0.7, repetition_penalty=1.2):
        self.eval()
        
        # CRITICAL FIX 3: pad=False! 
        # Now the AI actually reads your prompt instead of reading zeros!
        ids = tokenizer.encode(prompt, max_length=self.max_length, pad=False)
        input_ids = torch.tensor([ids], dtype=torch.long, device=next(self.parameters()).device)

        generated = []
        with torch.no_grad():
            for _ in range(max_new_tokens):
                current = input_ids[:, -self.max_length:]
                logits = self(current)

                next_logits = logits[:, -1, :] / temperature

                if repetition_penalty != 1.0:
                    for token_id in set(input_ids[0].tolist() + generated):
                        if next_logits[0, token_id] > 0:
                            next_logits[0, token_id] /= repetition_penalty
                        else:
                            next_logits[0, token_id] *= repetition_penalty

                probs = torch.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

                if next_token.item() == 2: # Stop if END token
                    break

                generated.append(next_token.item())
                input_ids = torch.cat([input_ids, next_token], dim=1)

        return tokenizer.decode(generated)

print("✅ All AI classes defined!")
print("   Ready to create your model in the next cell")

In [ ]:
# CELL 4 — CREATE YOUR AI MODEL
# This actually builds the brain

# You will add vocabulary from training data in Phase 3
# For now create with expanded vocab — will grow further
tokenizer = SimpleTokenizer()

# ══════════════════════════════════════════════════
# EXPANDED BASIC WORDS — v3.0
# Includes Gemini recommended bridge words
# These help AI understand English connectors
# not just Godot keywords
# ══════════════════════════════════════════════════

basic_words = """
extends node scene player enemy coin jump move speed gravity
health score level game godot gdscript characterbody2d
rigidbody2d area2d collisionshape2d sprite2d animatedsprite2d
tilemap canvaslayer label button input velocity position
signal func var const export ready process physics
android mobile touch screen apk export build
platformer shooter puzzle rpg topdown runner
design mechanic difficulty balance feel polish
audio sound music sfx animation sprite pixel

# CORE GAME DEV WORDS
extends class_name node scene player enemy npc boss
health stamina damage score level xp
checkpoint respawn save load pause menu ui hud

# GODOT 4 CORE NODES
Node Node2D CharacterBody2D StaticBody2D RigidBody2D
Area2D CollisionShape2D CollisionPolygon2D
Sprite2D AnimatedSprite2D Camera2D TileMap TileSet
CanvasLayer Control Label Button TextureButton
ProgressBar Timer AudioStreamPlayer2D AudioStreamPlayer
GPUParticles2D CPUParticles2D RayCast2D Line2D
ParallaxBackground ParallaxLayer VisibleOnScreenNotifier2D
VBoxContainer HBoxContainer MarginContainer CenterContainer
PanelContainer ScrollContainer TabContainer
TouchScreenButton VirtualKeyboard LineEdit

# GDSCRIPT FUNDAMENTALS
func var const enum static
if elif else match
for while break continue pass return
true false null
await yield signal
onready export

# INPUT AND MOVEMENT
Input InputEvent InputMap
move_and_slide move_and_collide
velocity direction delta
jump gravity acceleration friction
dash double_jump coyote_time

# MOBILE AND ANDROID
android mobile touch screen apk export
portrait landscape stretch_mode stretch_aspect
viewport scaling dpi
InputEventScreenTouch InputEventScreenDrag

# PERFORMANCE
delta fps process physics_process
queue_free object_pooling
visible_on_screen cull_mask
low_processor_mode
compression atlas batching

# GAME SYSTEMS
autoload singleton save_system load_system
game_manager state_machine fsm
scene_transition checkpoint respawn
inventory shop upgrade system

# DESIGN AND FEEL
coyote_time screen_shake hitstop
camera_zoom tween easing lerp
particles feedback sfx bgm
combo cooldown timer multiplier

# DATA TYPES AND STRUCTURES
int float bool String StringName
Array Dictionary PackedScene
Vector2 Rect2 Color Transform2D
signal Callable Resource

# BRIDGE WORDS — Gemini recommendation
# These help AI form proper English sentences
should because after before instead
attach handle control connect detect
return remove spawn reload reset
belongs requires whenever wherever
between without through during
another together separate individual
automatically immediately gradually
properly correctly accurately reliably
increase decrease subtract multiply
enable disable toggle activate
distance direction magnitude normalize
instance inherit extend override
reference pointer access retrieve
trigger emit receive respond listen
update refresh rebuild recreate
combine separate merge split filter
"""

tokenizer.add_text(basic_words)

# Create your AI
print("\n🧠 Creating YOUR AI model...")
your_ai = YourGameAI(
    vocab_size=tokenizer.vocab_size(),
    embed_dim=512,     # memory per word
    num_layers=8,      # thinking layers
    num_heads=8,       # attention heads
    ff_dim=512,        # feedforward width — matches embed_dim
    max_length=512,    # max text length
    dropout=0.2        # prevents overfitting
)

your_ai = your_ai.to(device)
print(f"✅ YOUR AI MODEL CREATED!")
print(f"   Running on: {device}")
print(f"   Vocabulary size: {tokenizer.vocab_size()} words")
print(f"   Parameters: {sum(p.numel() for p in your_ai.parameters()):,}")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║         CELL 5 — THE ULTIMATE GODOT 4 DATASET            ║
# ║  Rewritten with PURE CODE and 200+ New Unique Examples   ║
# ╚══════════════════════════════════════════════════════════╝

training_data = [
    # ─── REWRITTEN ORIGINAL EXAMPLES (NOW WITH REAL CODE) ───
    {"q": "how do i make a player jump in godot 4", "a": "if Input.is_action_just_pressed('jump') and is_on_floor():\n    velocity.y = JUMP_VELOCITY"},
    {"q": "how do i make top down movement in godot 4", "a": "var direction = Input.get_vector('ui_left', 'ui_right', 'ui_up', 'ui_down')\nvelocity = direction * SPEED\nmove_and_slide()"},
    {"q": "how do i make a player dash in godot 4", "a": "velocity = direction * DASH_SPEED\nawait get_tree().create_timer(0.2).timeout\nvelocity = direction * NORMAL_SPEED"},
    {"q": "how do i detect if a player enters an area", "a": "func _on_body_entered(body):\n    if body.is_in_group('player'):\n        print('Player entered!')"},
    {"q": "how do i change scenes in godot 4", "a": "get_tree().change_scene_to_file('res://level2.tscn')"},
    {"q": "how do i reload the current scene", "a": "get_tree().reload_current_scene()"},
    {"q": "how do i quit the game", "a": "get_tree().quit()"},
    {"q": "how do i pause the game", "a": "get_tree().paused = true"},
    {"q": "how do i unpause the game", "a": "get_tree().paused = false"},
    {"q": "how do i make a timer in code", "a": "await get_tree().create_timer(1.5).timeout"},
    {"q": "how do i flip a sprite horizontally", "a": "$Sprite2D.flip_h = true"},
    {"q": "how do i play a sound", "a": "$AudioStreamPlayer.play()"},
    {"q": "how do i hide a node", "a": "hide()"},
    {"q": "how do i show a node", "a": "show()"},
    {"q": "how do i destroy a node", "a": "queue_free()"},
    {"q": "how do i get the parent node", "a": "var parent = get_parent()"},
    {"q": "how do i add a child node", "a": "var enemy = enemy_scene.instantiate()\nadd_child(enemy)"},

    # ─── NEW: INPUT HANDLING ───
    {"q": "how to get mouse position", "a": "var mouse_pos = get_global_mouse_position()"},
    {"q": "how to check if mouse button is clicked", "a": "if Input.is_mouse_button_pressed(MOUSE_BUTTON_LEFT):\n    shoot()"},
    {"q": "how to hide the mouse cursor", "a": "Input.set_mouse_mode(Input.MOUSE_MODE_HIDDEN)"},
    {"q": "how to lock the mouse to the screen", "a": "Input.set_mouse_mode(Input.MOUSE_MODE_CAPTURED)"},
    {"q": "how to get joypad axis", "a": "var left_stick_x = Input.get_joy_axis(0, JOY_AXIS_LEFT_X)"},
    {"q": "how to check if a specific key is pressed", "a": "if Input.is_key_pressed(KEY_SPACE):\n    jump()"},
    {"q": "how to get input direction as an integer", "a": "var dir = Input.get_axis('move_left', 'move_right')"},

    # ─── NEW: MATH AND NUMBERS ───
    {"q": "how to generate a random float", "a": "var rand_val = randf()"},
    {"q": "how to generate a random float between two numbers", "a": "var rand_val = randf_range(1.5, 5.5)"},
    {"q": "how to generate a random integer", "a": "var rand_int = randi() % 10"},
    {"q": "how to generate a random integer between two numbers", "a": "var rand_int = randi_range(1, 100)"},
    {"q": "how to clamp a number", "a": "health = clamp(health, 0, max_health)"},
    {"q": "how to lerp a number", "a": "speed = lerp(speed, target_speed, 0.1)"},
    {"q": "how to move a number toward another number", "a": "speed = move_toward(speed, 0.0, friction * delta)"},
    {"q": "how to get the absolute value", "a": "var positive_val = abs(-50)"},
    {"q": "how to round a number", "a": "var rounded = round(3.7)"},
    {"q": "how to round down a number", "a": "var floored = floor(3.7)"},
    {"q": "how to round up a number", "a": "var ceiling = ceil(3.2)"},
    {"q": "how to get the sign of a number", "a": "var direction = sign(velocity.x)"},
    {"q": "how to wrap a number around", "a": "var angle = wrapf(current_angle, 0.0, 360.0)"},

    # ─── NEW: VECTORS AND MOVEMENT ───
    {"q": "how to get distance between two points", "a": "var dist = position.distance_to(target_position)"},
    {"q": "how to get direction to a target", "a": "var dir = position.direction_to(target_position)"},
    {"q": "how to normalize a vector", "a": "velocity = velocity.normalized() * speed"},
    {"q": "how to lerp a vector2", "a": "position = position.lerp(target_pos, 0.1)"},
    {"q": "how to move a vector2 toward a target", "a": "position = position.move_toward(target_pos, speed * delta)"},
    {"q": "how to get the length of a vector", "a": "var speed = velocity.length()"},
    {"q": "how to reflect a vector off a wall", "a": "velocity = velocity.bounce(collision_normal)"},
    {"q": "how to rotate a vector2", "a": "var rotated_vec = my_vector.rotated(deg_to_rad(90))"},
    {"q": "how to get the dot product of two vectors", "a": "var dot = vec1.dot(vec2)"},
    {"q": "how to get the cross product of two vectors", "a": "var cross = vec1.cross(vec2)"},

    # ─── NEW: NODE AND SCENE TREE MANIPULATION ───
    {"q": "how to find a node by name", "a": "var player = get_node('Player')"},
    {"q": "how to find a node using shorthand", "a": "var player = $Player"},
    {"q": "how to get all nodes in a group", "a": "var enemies = get_tree().get_nodes_in_group('enemies')"},
    {"q": "how to add a node to a group", "a": "add_to_group('enemies')"},
    {"q": "how to remove a node from a group", "a": "remove_from_group('enemies')"},
    {"q": "how to check if a node is in a group", "a": "if is_in_group('player'):\n    take_damage()"},
    {"q": "how to call a function on all nodes in a group", "a": "get_tree().call_group('enemies', 'take_damage', 10)"},
    {"q": "how to get the root node", "a": "var root = get_tree().root"},
    {"q": "how to get the number of children", "a": "var count = get_child_count()"},
    {"q": "how to get a specific child by index", "a": "var first_child = get_child(0)"},
    {"q": "how to move a child node to the bottom", "a": "move_child(my_node, -1)"},
    {"q": "how to duplicate a node", "a": "var copy = my_node.duplicate()\nadd_child(copy)"},

    # ─── NEW: SIGNALS AND EVENTS ───
    {"q": "how to declare a custom signal", "a": "signal health_changed(new_health)"},
    {"q": "how to emit a custom signal", "a": "health_changed.emit(current_health)"},
    {"q": "how to connect a signal in code", "a": "$Button.pressed.connect(_on_button_pressed)"},
    {"q": "how to disconnect a signal", "a": "$Button.pressed.disconnect(_on_button_pressed)"},
    {"q": "how to check if a signal is connected", "a": "if $Button.pressed.is_connected(_on_button_pressed):\n    pass"},
    {"q": "how to wait for a signal to emit", "a": "await $Timer.timeout"},

    # ─── NEW: ARRAYS AND DICTIONARIES ───
    {"q": "how to create an empty array", "a": "var my_array = []"},
    {"q": "how to add an item to an array", "a": "my_array.append('sword')"},
    {"q": "how to remove an item from an array", "a": "my_array.erase('sword')"},
    {"q": "how to remove an item at a specific index", "a": "my_array.remove_at(0)"},
    {"q": "how to get the size of an array", "a": "var size = my_array.size()"},
    {"q": "how to clear an array", "a": "my_array.clear()"},
    {"q": "how to check if an array contains an item", "a": "if my_array.has('key'):\n    open_door()"},
    {"q": "how to sort an array", "a": "my_array.sort()"},
    {"q": "how to shuffle an array randomly", "a": "my_array.shuffle()"},
    {"q": "how to get a random item from an array", "a": "var random_item = my_array.pick_random()"},
    {"q": "how to create a dictionary", "a": "var stats = {'hp': 100, 'mp': 50}"},
    {"q": "how to add a key to a dictionary", "a": "stats['strength'] = 10"},
    {"q": "how to check if a dictionary has a key", "a": "if stats.has('hp'):\n    print(stats['hp'])"},
    {"q": "how to get all keys in a dictionary", "a": "var keys = stats.keys()"},
    {"q": "how to get all values in a dictionary", "a": "var values = stats.values()"},

    # ─── NEW: STRINGS AND TEXT ───
    {"q": "how to combine strings", "a": "var full_name = first_name + ' ' + last_name"},
    {"q": "how to convert a number to a string", "a": "var text = str(100)"},
    {"q": "how to convert a string to an integer", "a": "var num = '50'.to_int()"},
    {"q": "how to convert a string to a float", "a": "var num = '3.14'.to_float()"},
    {"q": "how to make a string uppercase", "a": "var big = my_string.to_upper()"},
    {"q": "how to make a string lowercase", "a": "var small = my_string.to_lower()"},
    {"q": "how to replace text in a string", "a": "var new_text = old_text.replace('bad', 'good')"},
    {"q": "how to split a string into an array", "a": "var words = sentence.split(' ')"},
    {"q": "how to format a string with variables", "a": "var text = 'Health: %d' % current_health"},

    # ─── NEW: UI AND CONTROL NODES ───
    {"q": "how to change label text", "a": "$Label.text = 'Score: ' + str(score)"},
    {"q": "how to change button text", "a": "$Button.text = 'Start Game'"},
    {"q": "how to disable a button", "a": "$Button.disabled = true"},
    {"q": "how to update a progress bar", "a": "$ProgressBar.value = current_health"},
    {"q": "how to set progress bar max value", "a": "$ProgressBar.max_value = 100"},
    {"q": "how to get text from a line edit", "a": "var player_name = $LineEdit.text"},
    {"q": "how to clear a line edit", "a": "$LineEdit.clear()"},
    {"q": "how to grab focus on a UI element", "a": "$Button.grab_focus()"},
    {"q": "how to change UI modulate color", "a": "$Panel.modulate = Color(1, 0, 0, 1) # Red"},

    # ─── NEW: SAVING AND LOADING FILES ───
    {"q": "how to save text to a file", "a": "var file = FileAccess.open('user://save.txt', FileAccess.WRITE)\nfile.store_string('Hello')\nfile.close()"},
    {"q": "how to read text from a file", "a": "var file = FileAccess.open('user://save.txt', FileAccess.READ)\nvar text = file.get_as_text()\nfile.close()"},
    {"q": "how to check if a file exists", "a": "if FileAccess.file_exists('user://save.txt'):\n    load_game()"},
    {"q": "how to convert a dictionary to JSON", "a": "var json_string = JSON.stringify(my_dict)"},
    {"q": "how to parse JSON into a dictionary", "a": "var my_dict = JSON.parse_string(json_string)"},

    # ─── NEW: PHYSICS AND COLLISIONS ───
    {"q": "how to check if a raycast is colliding", "a": "if $RayCast2D.is_colliding():\n    print('Hit!')"},
    {"q": "how to get the object a raycast hit", "a": "var target = $RayCast2D.get_collider()"},
    {"q": "how to get the exact point a raycast hit", "a": "var hit_pos = $RayCast2D.get_collision_point()"},
    {"q": "how to get the normal of a raycast hit", "a": "var normal = $RayCast2D.get_collision_normal()"},
    {"q": "how to force a raycast to update immediately", "a": "$RayCast2D.force_raycast_update()"},
    {"q": "how to disable a collision shape", "a": "$CollisionShape2D.set_deferred('disabled', true)"},
    {"q": "how to enable a collision shape", "a": "$CollisionShape2D.set_deferred('disabled', false)"},
    {"q": "how to change collision layer", "a": "collision_layer = 2"},
    {"q": "how to change collision mask", "a": "collision_mask = 4"},

    # ─── NEW: ANIMATION AND TWEENS ───
    {"q": "how to play an animation", "a": "$AnimationPlayer.play('walk')"},
    {"q": "how to play an animation backwards", "a": "$AnimationPlayer.play_backwards('walk')"},
    {"q": "how to stop an animation", "a": "$AnimationPlayer.stop()"},
    {"q": "how to check if an animation is playing", "a": "if $AnimationPlayer.is_playing():\n    pass"},
    {"q": "how to change animation speed", "a": "$AnimationPlayer.speed_scale = 1.5"},
    {"q": "how to create a tween", "a": "var tween = create_tween()"},
    {"q": "how to tween a property", "a": "var tween = create_tween()\ntween.tween_property(self, 'position', target_pos, 1.0)"},
    {"q": "how to make tweens play at the same time", "a": "var tween = create_tween().set_parallel(true)\ntween.tween_property(self, 'scale', Vector2(2,2), 1.0)"},
    {"q": "how to loop a tween", "a": "var tween = create_tween().set_loops()\ntween.tween_property(self, 'rotation', 360.0, 2.0)"},

    # ─── NEW: 3D SPECIFIC ───
    {"q": "how to move a character in 3d", "a": "var dir = (transform.basis * Vector3(input.x, 0, input.y)).normalized()\nvelocity.x = dir.x * speed\nvelocity.z = dir.z * speed\nmove_and_slide()"},
    {"q": "how to rotate a 3d object", "a": "rotate_y(deg_to_rad(45))"},
    {"q": "how to look at a 3d target", "a": "look_at(target.global_position, Vector3.UP)"},
    {"q": "how to get 3d forward direction", "a": "var forward = -transform.basis.z"},
    {"q": "how to get 3d right direction", "a": "var right = transform.basis.x"},
    {"q": "how to get 3d up direction", "a": "var up = transform.basis.y"},
    {"q": "how to cast a 3d raycast", "a": "if $RayCast3D.is_colliding():\n    var obj = $RayCast3D.get_collider()"},
    {"q": "how to hide the mouse in 3d", "a": "Input.set_mouse_mode(Input.MOUSE_MODE_CAPTURED)"},

    # ─── NEW: AUDIO AND PARTICLES ───
    {"q": "how to change audio volume", "a": "$AudioStreamPlayer.volume_db = -10.0"},
    {"q": "how to change audio pitch", "a": "$AudioStreamPlayer.pitch_scale = 1.2"},
    {"q": "how to emit particles", "a": "$GPUParticles2D.emitting = true"},
    {"q": "how to stop emitting particles", "a": "$GPUParticles2D.emitting = false"},
    {"q": "how to restart particles", "a": "$GPUParticles2D.restart()"},

    # ─── NEW: ENGINE AND OS ───
    {"q": "how to get the current fps", "a": "var fps = Engine.get_frames_per_second()"},
    {"q": "how to open a web link", "a": "OS.shell_open('https://godotengine.org')"},
    {"q": "how to get the operating system name", "a": "var os_name = OS.get_name()"},
    {"q": "how to copy text to clipboard", "a": "DisplayServer.clipboard_set('Hello')"},
    {"q": "how to get text from clipboard", "a": "var text = DisplayServer.clipboard_get()"},
    {"q": "how to change window title", "a": "DisplayServer.window_set_title('My Awesome Game')"}
]

print(f"✅ MASSIVE DATASET LOADED!")
print(f"   Total Examples: {len(training_data)}")

In [ ]:
# Create a dedicated list for your new examples
new_godot_knowledge = [
    {"q": "how to change scenes", "a": "get_tree().change_scene_to_file('res://levels/level_2.tscn')"},
    {"q": "how to make a timer", "a": "await get_tree().create_timer(2.0).timeout\nprint('Time is up!')"},
    {"q": "how to detect collision", "a": "func _on_body_entered(body):\n    if body.is_in_group('player'):\n        queue_free()"},
    {"q": "how to flip a sprite", "a": "$Sprite2D.flip_h = velocity.x < 0"},
    {"q": "how to play a sound", "a": "$AudioStreamPlayer.play()"},
    {"q": "how to use delta", "a": "position += velocity * delta"},
    {"q": "how to get mouse position", "a": "var mouse_pos = get_global_mouse_position()"},
    {"q": "how to add a child node", "a": "var enemy = enemy_scene.instantiate()\nadd_child(enemy)"},
    {"q": "how to check for input", "a": "if Input.is_action_just_pressed('jump'):\n    jump()"},
    {"q": "how to quit game", "a": "get_tree().quit()"}
]

# Add them to the main training list
training_data.extend(new_godot_knowledge)

print(f"✅ Data added! You now have {len(training_data)} total examples.")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║         NEW GODOT 4 CODE EXAMPLES (SHORT & PRECISE)      ║
# ╚══════════════════════════════════════════════════════════╝

even_more_godot_knowledge = [
    # --- SCENE & NODE MANAGEMENT ---
    {"q": "how to reload current scene", "a": "get_tree().reload_current_scene()"},
    {"q": "how to get the parent node", "a": "get_parent()"},
    {"q": "how to find all enemies in a group", "a": "var enemies = get_tree().get_nodes_in_group('enemy')"},
    {"q": "how to hide a node", "a": "hide()"},
    {"q": "how to show a node", "a": "show()"},
    {"q": "how to destroy a node safely", "a": "queue_free()"},
    
    # --- MOVEMENT & PHYSICS ---
    {"q": "how to apply gravity", "a": "velocity.y += gravity * delta"},
    {"q": "how to get horizontal input direction", "a": "var direction = Input.get_axis('ui_left', 'ui_right')"},
    {"q": "how to check if player is on the floor", "a": "if is_on_floor():\n    pass"},
    {"q": "how to check if player is hitting a wall", "a": "if is_on_wall():\n    pass"},
    {"q": "how to move and collide", "a": "move_and_slide()"},
    
    # --- UI & TEXT ---
    {"q": "how to change label text", "a": "$Label.text = 'Hello World'"},
    {"q": "how to change a button text", "a": "$Button.text = 'Click Me'"},
    {"q": "how to hide the mouse cursor", "a": "Input.set_mouse_mode(Input.MOUSE_MODE_HIDDEN)"},
    
    # --- SIGNALS & TIMERS ---
    {"q": "how to wait 1 second in code", "a": "await get_tree().create_timer(1.0).timeout"},
    {"q": "how to connect a button press via code", "a": "$Button.pressed.connect(_on_button_pressed)"},
    {"q": "how to emit a custom signal", "a": "emit_signal('my_custom_signal')"},
    
    # --- ANIMATION & AUDIO ---
    {"q": "how to play an animation", "a": "$AnimationPlayer.play('run')"},
    {"q": "how to stop an animation", "a": "$AnimationPlayer.stop()"},
    {"q": "how to play a sound effect", "a": "$AudioStreamPlayer.play()"},
    {"q": "how to flip a sprite horizontally", "a": "$Sprite2D.flip_h = true"},
    
    # --- MATH & LOGIC ---
    {"q": "how to generate a random number", "a": "var random_val = randf_range(0.0, 10.0)"},
    {"q": "how to get distance between two nodes", "a": "var dist = global_position.distance_to(target.global_position)"},
    {"q": "how to print a variable to console", "a": "print('The value is: ', my_variable)"},
    {"q": "how to clamp a number between 0 and 100", "a": "var health = clamp(health, 0, 100)"},
    
    # --- GAME STATE ---
    {"q": "how to pause the game", "a": "get_tree().paused = true"},
    {"q": "how to unpause the game", "a": "get_tree().paused = false"},
    {"q": "how to quit the game to desktop", "a": "get_tree().quit()"},
    {"q": "how to save a file", "a": "var file = FileAccess.open('user://save.dat', FileAccess.WRITE)"},
    {"q": "how to load a file", "a": "var file = FileAccess.open('user://save.dat', FileAccess.READ)"}
]

# Add these new examples to your main training list
training_data.extend(even_more_godot_knowledge)

print(f"✅ SUCCESS! Added 30 new short code examples.")
print(f"🧠 Your AI now has {len(training_data)} total examples to learn from!")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║         CELL 5.5 — 300 NEW ADVANCED EXAMPLES             ║
# ║  Run this cell to add these to your AI's training data!  ║
# ╚══════════════════════════════════════════════════════════╝

new_advanced_data = [
    # --- ADVANCED ARRAYS ---
    {"q": "how to get the last item in an array", "a": "var last = my_array[-1]"},
    {"q": "how to get the first item in an array", "a": "var first = my_array.front()"},
    {"q": "how to remove and return the last item of an array", "a": "var item = my_array.pop_back()"},
    {"q": "how to remove and return the first item of an array", "a": "var item = my_array.pop_front()"},
    {"q": "how to add an item to the start of an array", "a": "my_array.push_front('new_item')"},
    {"q": "how to reverse an array", "a": "my_array.reverse()"},
    {"q": "how to find the maximum value in an array", "a": "var highest = my_array.max()"},
    {"q": "how to find the minimum value in an array", "a": "var lowest = my_array.min()"},
    {"q": "how to count how many times an item appears in an array", "a": "var count = my_array.count('apple')"},
    {"q": "how to duplicate an array", "a": "var copy = my_array.duplicate()"},
    {"q": "how to fill an array with the same value", "a": "my_array.fill(0)"},
    {"q": "how to slice an array", "a": "var sub_array = my_array.slice(1, 4)"},
    {"q": "how to check if all items in array are true", "a": "var is_all_true = my_array.all(func(x): return x == true)"},
    {"q": "how to check if any item in array is true", "a": "var has_true = my_array.any(func(x): return x == true)"},
    {"q": "how to filter an array", "a": "var evens = my_array.filter(func(x): return x % 2 == 0)"},
    {"q": "how to map an array to new values", "a": "var doubled = my_array.map(func(x): return x * 2)"},
    {"q": "how to do a custom sort on an array", "a": "my_array.sort_custom(func(a, b): return a.health > b.health)"},
    {"q": "how to binary search an array", "a": "var index = my_array.bsearch(10)"},

    # --- ADVANCED DICTIONARIES ---
    {"q": "how to clear a dictionary", "a": "my_dict.clear()"},
    {"q": "how to remove a key from a dictionary", "a": "my_dict.erase('score')"},
    {"q": "how to check if a dictionary is empty", "a": "if my_dict.is_empty():\n    pass"},
    {"q": "how to get a value or add it if it doesnt exist", "a": "var val = my_dict.get_or_add('coins', 0)"},
    {"q": "how to merge two dictionaries", "a": "dict1.merge(dict2)"},
    {"q": "how to find a key by its value", "a": "var key = my_dict.find_key('John')"},
    {"q": "how to duplicate a dictionary", "a": "var copy = my_dict.duplicate()"},
    {"q": "how to hash a dictionary", "a": "var hash_val = my_dict.hash()"},

    # --- ADVANCED STRINGS ---
    {"q": "how to check if a string starts with text", "a": "if my_string.begins_with('Hello'):\n    pass"},
    {"q": "how to check if a string ends with text", "a": "if my_string.ends_with('.png'):\n    pass"},
    {"q": "how to remove a prefix from a string", "a": "var clean = my_string.trim_prefix('res://')"},
    {"q": "how to remove a suffix from a string", "a": "var clean = my_string.trim_suffix('.tscn')"},
    {"q": "how to pad a string with zeros on the left", "a": "var padded = str(score).lpad(5, '0')"},
    {"q": "how to pad a string on the right", "a": "var padded = my_string.rpad(10, ' ')"},
    {"q": "how to find text inside a string", "a": "var index = my_string.find('apple')"},
    {"q": "how to find text from the end of a string", "a": "var index = my_string.rfind('apple')"},
    {"q": "how to count occurrences in a string", "a": "var count = my_string.count('a')"},
    {"q": "how to get a slice of a string", "a": "var part = my_string.get_slice(',', 1)"},
    {"q": "how to join an array of strings", "a": "var sentence = ', '.join(string_array)"},
    {"q": "how to strip whitespace from a string", "a": "var clean = my_string.strip_edges()"},
    {"q": "how to get the md5 hash of a string", "a": "var hash = my_string.md5_text()"},
    {"q": "how to get the sha256 hash of a string", "a": "var hash = my_string.sha256_text()"},
    {"q": "how to check if a string is a valid integer", "a": "if my_string.is_valid_int():\n    pass"},
    {"q": "how to check if a string is a valid float", "a": "if my_string.is_valid_float():\n    pass"},

    # --- ADVANCED MATH ---
    {"q": "how to pingpong a value", "a": "var bounce = pingpong(time, 5.0)"},
    {"q": "how to get the positive modulo", "a": "var mod = posmod(angle, 360.0)"},
    {"q": "how to snap a number to a grid", "a": "var snapped_val = snapped(position.x, 32.0)"},
    {"q": "how to smoothstep a value", "a": "var smooth = smoothstep(0.0, 1.0, weight)"},
    {"q": "how to cubic interpolate", "a": "var val = cubic_interpolate(p0, p1, p2, p3, weight)"},
    {"q": "how to convert degrees to radians", "a": "var rad = deg_to_rad(90.0)"},
    {"q": "how to convert radians to degrees", "a": "var deg = rad_to_deg(PI)"},
    {"q": "how to get the square root", "a": "var root = sqrt(25.0)"},
    {"q": "how to raise a number to a power", "a": "var power = pow(2.0, 3.0)"},
    {"q": "how to get the sine of an angle", "a": "var s = sin(angle)"},
    {"q": "how to get the cosine of an angle", "a": "var c = cos(angle)"},
    {"q": "how to get the tangent of an angle", "a": "var t = tan(angle)"},
    {"q": "how to get the arc tangent", "a": "var angle = atan2(dir.y, dir.x)"},
    {"q": "how to move toward an angle safely", "a": "var new_angle = rotate_toward(current, target, speed * delta)"},
    {"q": "how to lerp an angle safely", "a": "var new_angle = lerp_angle(current, target, 0.1)"},

    # --- COLORS ---
    {"q": "how to create a color from hex", "a": "var my_color = Color.html('#ff0000')"},
    {"q": "how to create a color from hsv", "a": "var my_color = Color.from_hsv(0.5, 1.0, 1.0)"},
    {"q": "how to invert a color", "a": "var inv = my_color.inverted()"},
    {"q": "how to lighten a color", "a": "var light = my_color.lightened(0.2)"},
    {"q": "how to darken a color", "a": "var dark = my_color.darkened(0.2)"},
    {"q": "how to blend two colors", "a": "var blended = color1.blend(color2)"},
    {"q": "how to lerp a color", "a": "var new_color = color1.lerp(color2, 0.5)"},
    {"q": "how to get the luminance of a color", "a": "var lum = my_color.get_luminance()"},

    # --- RANDOMNESS ---
    {"q": "how to set the random seed", "a": "seed(12345)"},
    {"q": "how to randomize the seed based on time", "a": "randomize()"},
    {"q": "how to get a random float with normal distribution", "a": "var rand = randfn(0.0, 1.0)"},
    {"q": "how to get a random point inside a circle", "a": "var point = randv_circle() * radius"},

    # --- INPUT MAP (RUNTIME) ---
    {"q": "how to add a new input action in code", "a": "InputMap.add_action('new_jump')"},
    {"q": "how to assign a key to an input action in code", "a": "var ev = InputEventKey.new()\nev.keycode = KEY_SPACE\nInputMap.action_add_event('new_jump', ev)"},
    {"q": "how to erase all events from an action", "a": "InputMap.action_erase_events('jump')"},
    {"q": "how to get all events for an action", "a": "var events = InputMap.action_get_events('jump')"},
    {"q": "how to check if an action exists", "a": "if InputMap.has_action('jump'):\n    pass"},
    {"q": "how to vibrate the controller", "a": "Input.start_joy_vibration(0, 0.5, 0.5, 1.0)"},
    {"q": "how to stop controller vibration", "a": "Input.stop_joy_vibration(0)"},
    {"q": "how to change the mouse cursor image", "a": "Input.set_custom_mouse_cursor(preload('res://cursor.png'))"},

    # --- NODE MANAGEMENT ---
    {"q": "how to check if a node is an ancestor of another", "a": "if node_a.is_ancestor_of(node_b):\n    pass"},
    {"q": "how to turn off process for a node", "a": "set_process(false)"},
    {"q": "how to turn off physics process for a node", "a": "set_physics_process(false)"},
    {"q": "how to get the index of a child node", "a": "var idx = my_node.get_index()"},
    {"q": "how to reparent a node", "a": "my_node.reparent(new_parent)"},
    {"q": "how to check if a node is queued for deletion", "a": "if my_node.is_queued_for_deletion():\n    pass"},
    {"q": "how to set a node to process even when paused", "a": "process_mode = Node.PROCESS_MODE_ALWAYS"},
    {"q": "how to set a node to stop when paused", "a": "process_mode = Node.PROCESS_MODE_PAUSABLE"},
    {"q": "how to get the current frame number", "a": "var frame = Engine.get_process_frames()"},
    {"q": "how to get the physics delta time", "a": "var dt = get_physics_process_delta_time()"},

    # --- 2D PHYSICS (RIGIDBODY & FORCES) ---
    {"q": "how to apply an impulse to a rigidbody2d", "a": "$RigidBody2D.apply_central_impulse(Vector2(0, -500))"},
    {"q": "how to apply an impulse at a specific point", "a": "$RigidBody2D.apply_impulse(Vector2(0, -500), Vector2(10, 0))"},
    {"q": "how to apply torque to a rigidbody2d", "a": "$RigidBody2D.apply_torque(1000.0)"},
    {"q": "how to add a constant force to a rigidbody2d", "a": "$RigidBody2D.add_constant_force(Vector2(100, 0))"},
    {"q": "how to clear constant forces", "a": "$RigidBody2D.constant_force = Vector2.ZERO"},
    {"q": "how to set the center of mass", "a": "$RigidBody2D.center_of_mass = Vector2(0, 10)"},
    {"q": "how to sleep a rigidbody2d", "a": "$RigidBody2D.sleeping = true"},
    {"q": "how to wake up a rigidbody2d", "a": "$RigidBody2D.sleeping = false"},
    {"q": "how to change gravity scale on rigidbody2d", "a": "$RigidBody2D.gravity_scale = 2.0"},
    {"q": "how to get the linear velocity of a rigidbody2d", "a": "var vel = $RigidBody2D.linear_velocity"},
    {"q": "how to get the angular velocity of a rigidbody2d", "a": "var ang_vel = $RigidBody2D.angular_velocity"},

    # --- 3D PHYSICS (RIGIDBODY & FORCES) ---
    {"q": "how to apply an impulse to a rigidbody3d", "a": "$RigidBody3D.apply_central_impulse(Vector3(0, 10, 0))"},
    {"q": "how to apply torque to a rigidbody3d", "a": "$RigidBody3D.apply_torque(Vector3(0, 5, 0))"},
    {"q": "how to lock rotation on a rigidbody3d", "a": "$RigidBody3D.axis_lock_angular_x = true"},
    {"q": "how to lock linear movement on a rigidbody3d", "a": "$RigidBody3D.axis_lock_linear_y = true"},
    {"q": "how to cast a shape cast in 3d", "a": "if $ShapeCast3D.is_colliding():\n    pass"},
    {"q": "how to get the collision normal in 3d", "a": "var normal = $RayCast3D.get_collision_normal()"},

    # --- TILEMAPS (GODOT 4) ---
    {"q": "how to set a tile in a tilemaplayer", "a": "$TileMapLayer.set_cell(Vector2i(5, 5), 0, Vector2i(1, 1))"},
    {"q": "how to erase a tile in a tilemaplayer", "a": "$TileMapLayer.erase_cell(Vector2i(5, 5))"},
    {"q": "how to get the source id of a tile", "a": "var id = $TileMapLayer.get_cell_source_id(Vector2i(5, 5))"},
    {"q": "how to get the atlas coords of a tile", "a": "var coords = $TileMapLayer.get_cell_atlas_coords(Vector2i(5, 5))"},
    {"q": "how to convert local position to map coordinates", "a": "var map_pos = $TileMapLayer.local_to_map(global_position)"},
    {"q": "how to convert map coordinates to local position", "a": "var local_pos = $TileMapLayer.map_to_local(Vector2i(5, 5))"},
    {"q": "how to get all used cells in a tilemaplayer", "a": "var cells = $TileMapLayer.get_used_cells()"},
    {"q": "how to clear a tilemaplayer", "a": "$TileMapLayer.clear()"},

    # --- NAVIGATION ---
    {"q": "how to set the target position for a navigation agent", "a": "$NavigationAgent2D.target_position = player.global_position"},
    {"q": "how to get the next path position for navigation", "a": "var next_pos = $NavigationAgent2D.get_next_path_position()"},
    {"q": "how to check if navigation is finished", "a": "if $NavigationAgent2D.is_navigation_finished():\n    pass"},
    {"q": "how to check if navigation target is reachable", "a": "if $NavigationAgent2D.is_target_reachable():\n    pass"},
    {"q": "how to get the distance to the navigation target", "a": "var dist = $NavigationAgent2D.distance_to_target()"},
    {"q": "how to set navigation avoidance velocity", "a": "$NavigationAgent2D.set_velocity(desired_velocity)"},

    # --- AUDIO BUSES AND DSP ---
    {"q": "how to mute an audio bus", "a": "AudioServer.set_bus_mute(AudioServer.get_bus_index('Master'), true)"},
    {"q": "how to change the volume of an audio bus", "a": "AudioServer.set_bus_volume_db(AudioServer.get_bus_index('Music'), -15.0)"},
    {"q": "how to get the volume of an audio bus", "a": "var vol = AudioServer.get_bus_volume_db(AudioServer.get_bus_index('SFX'))"},
    {"q": "how to add an audio effect to a bus", "a": "AudioServer.add_bus_effect(bus_idx, AudioEffectReverb.new())"},
    {"q": "how to get the peak volume of a bus", "a": "var peak = AudioServer.get_bus_peak_volume_left_db(bus_idx, 0)"},
    {"q": "how to bypass all effects on an audio bus", "a": "AudioServer.set_bus_bypass_effects(bus_idx, true)"},

    # --- CAMERA 2D & 3D ---
    {"q": "how to make a camera the active one", "a": "$Camera2D.make_current()"},
    {"q": "how to get the screen center position of a camera2d", "a": "var center = $Camera2D.get_screen_center_position()"},
    {"q": "how to set camera zoom", "a": "$Camera2D.zoom = Vector2(2.0, 2.0)"},
    {"q": "how to set camera limits", "a": "$Camera2D.limit_left = -1000\n$Camera2D.limit_right = 1000"},
    {"q": "how to project a 3d point to 2d screen space", "a": "var screen_pos = $Camera3D.unproject_position(target.global_position)"},
    {"q": "how to project a 2d screen point to a 3d ray", "a": "var ray_dir = $Camera3D.project_ray_normal(mouse_pos)"},

    # --- UI AND CONTROL NODES (ADVANCED) ---
    {"q": "how to set anchors preset on a control", "a": "$Panel.set_anchors_preset(Control.PRESET_FULL_RECT)"},
    {"q": "how to get the minimum size of a control", "a": "var min_size = $Button.get_minimum_size()"},
    {"q": "how to force a control to update its layout", "a": "$VBoxContainer.queue_sort()"},
    {"q": "how to change the custom minimum size", "a": "$ColorRect.custom_minimum_size = Vector2(100, 100)"},
    {"q": "how to append text to a richtextlabel", "a": "$RichTextLabel.append_text('[color=red]Damage![/color]')"},
    {"q": "how to clear a richtextlabel", "a": "$RichTextLabel.clear()"},
    {"q": "how to add an item to an itemlist", "a": "$ItemList.add_item('Sword', preload('res://icon.png'))"},
    {"q": "how to clear an itemlist", "a": "$ItemList.clear()"},
    {"q": "how to get the selected items in an itemlist", "a": "var selected = $ItemList.get_selected_items()"},
    {"q": "how to add a tab to a tabcontainer", "a": "$TabContainer.add_child(new_panel)"},
    {"q": "how to change the current tab in a tabcontainer", "a": "$TabContainer.current_tab = 1"},

    # --- HTTP AND NETWORKING ---
    {"q": "how to make an http get request", "a": "$HTTPRequest.request('https://api.example.com/data')"},
    {"q": "how to make an http post request", "a": "var headers = ['Content-Type: application/json']\n$HTTPRequest.request('https://api.com', headers, HTTPClient.METHOD_POST, json_data)"},
    {"q": "how to cancel an http request", "a": "$HTTPRequest.cancel_request()"},
    {"q": "how to parse an http response body", "a": "var json = JSON.parse_string(body.get_string_from_utf8())"},
    {"q": "how to get the local ip address", "a": "var ip = IP.get_local_addresses()[0]"},

    # --- CONFIG FILES (INI) ---
    {"q": "how to create a config file", "a": "var config = ConfigFile.new()"},
    {"q": "how to set a value in a config file", "a": "config.set_value('Player', 'name', 'Hero')"},
    {"q": "how to get a value from a config file", "a": "var name = config.get_value('Player', 'name', 'Default')"},
    {"q": "how to save a config file", "a": "config.save('user://settings.cfg')"},
    {"q": "how to load a config file", "a": "var err = config.load('user://settings.cfg')"},
    {"q": "how to check if a section exists in a config file", "a": "if config.has_section('Player'):\n    pass"},
    {"q": "how to get all sections in a config file", "a": "var sections = config.get_sections()"},

    # --- DIRECTORY ACCESS ---
    {"q": "how to get all files in a directory", "a": "var files = DirAccess.get_files_at('user://')"},
    {"q": "how to get all folders in a directory", "a": "var dirs = DirAccess.get_directories_at('user://')"},
    {"q": "how to create a new directory", "a": "DirAccess.make_dir_absolute('user://new_folder')"},
    {"q": "how to create a directory and all parent directories", "a": "DirAccess.make_dir_recursive_absolute('user://folder/subfolder')"},
    {"q": "how to delete a file", "a": "DirAccess.remove_absolute('user://save.txt')"},
    {"q": "how to check if a directory exists", "a": "if DirAccess.dir_exists_absolute('user://folder'):\n    pass"},
    {"q": "how to copy a file", "a": "DirAccess.copy_absolute('user://old.txt', 'user://new.txt')"},
    {"q": "how to rename a file", "a": "DirAccess.rename_absolute('user://old.txt', 'user://new.txt')"},

    # --- IMAGES AND TEXTURES ---
    {"q": "how to create a new image", "a": "var img = Image.create(64, 64, false, Image.FORMAT_RGBA8)"},
    {"q": "how to set a pixel color on an image", "a": "img.set_pixel(10, 10, Color.RED)"},
    {"q": "how to get a pixel color from an image", "a": "var color = img.get_pixel(10, 10)"},
    {"q": "how to save an image as a png", "a": "img.save_png('user://screenshot.png')"},
    {"q": "how to convert an image to an imagetexture", "a": "var tex = ImageTexture.create_from_image(img)"},
    {"q": "how to get the image data from a texture", "a": "var img = my_texture.get_image()"},
    {"q": "how to get the width of a texture", "a": "var w = my_texture.get_width()"},

    # --- DISPLAY SERVER AND WINDOWS ---
    {"q": "how to set window to fullscreen", "a": "DisplayServer.window_set_mode(DisplayServer.WINDOW_MODE_FULLSCREEN)"},
    {"q": "how to set window to windowed mode", "a": "DisplayServer.window_set_mode(DisplayServer.WINDOW_MODE_WINDOWED)"},
    {"q": "how to get the screen size", "a": "var size = DisplayServer.screen_get_size()"},
    {"q": "how to get the window size", "a": "var size = DisplayServer.window_get_size()"},
    {"q": "how to set the window size", "a": "DisplayServer.window_set_size(Vector2i(1280, 720))"},
    {"q": "how to center the window on screen", "a": "var screen_center = DisplayServer.screen_get_position() + DisplayServer.screen_get_size() / 2\nDisplayServer.window_set_position(screen_center - DisplayServer.window_get_size() / 2)"},
    {"q": "how to show an alert dialog", "a": "DisplayServer.dialog_show('Error', 'Something went wrong!', ['OK'])"},
    {"q": "how to keep the screen on for mobile", "a": "DisplayServer.window_set_keep_screen_on(true)"},

    # --- RESOURCES ---
    {"q": "how to load a resource dynamically", "a": "var res = ResourceLoader.load('res://my_resource.tres')"},
    {"q": "how to save a resource", "a": "ResourceSaver.save(my_resource, 'user://saved_res.tres')"},
    {"q": "how to check if a resource exists", "a": "if ResourceLoader.exists('res://my_resource.tres'):\n    pass"},
    {"q": "how to duplicate a resource", "a": "var copy = my_resource.duplicate(true)"},

    # --- THREADING ---
    {"q": "how to create a thread", "a": "var thread = Thread.new()\nthread.start(my_function)"},
    {"q": "how to wait for a thread to finish", "a": "var result = thread.wait_to_finish()"},
    {"q": "how to check if a thread is alive", "a": "if thread.is_alive():\n    pass"},
    {"q": "how to use a mutex", "a": "mutex.lock()\n# do thread safe stuff\nmutex.unlock()"},
    {"q": "how to use the worker thread pool", "a": "WorkerThreadPool.add_task(my_function, true)"},

    # --- SHADERS AND MATERIALS ---
    {"q": "how to set a shader parameter in code", "a": "$Sprite2D.material.set_shader_parameter('outline_color', Color.RED)"},
    {"q": "how to get a shader parameter in code", "a": "var color = $Sprite2D.material.get_shader_parameter('outline_color')"},
    {"q": "how to create a new material in code", "a": "var mat = StandardMaterial3D.new()\nmat.albedo_color = Color.BLUE"},
    {"q": "how to make a material unique", "a": "$MeshInstance3D.material_override = $MeshInstance3D.material_override.duplicate()"},

        # --- MISCELLANEOUS ---
    {"q": "how to print an error message", "a": "printerr('This is an error!')"},
    {"q": "how to print a warning message", "a": "push_warning('This is a warning!')"},
    {"q": "how to get the godot version", "a": "var version = Engine.get_version_info()"},
    {"q": "how to check if code is running in the editor", "a": "if Engine.is_editor_hint():\n    pass"},
    {"q": "how to execute a command line program", "a": "var output = []\nOS.execute('ping', ['8.8.8.8'], output)"},
    {"q": "how to get environment variables", "a": "var path = OS.get_environment('PATH')"},
    {"q": "how to get the unique id of an object", "a": "var id = my_node.get_instance_id()"},
    {"q": "how to check if an instance is valid", "a": "if is_instance_valid(my_node):\n    pass"},
    {"q": "how to free an object immediately", "a": "my_node.free()"},
    {"q": "how to call a function by string name", "a": "my_node.call('take_damage', 10)"},
    {"q": "how to check if a node has a specific method", "a": "if my_node.has_method('take_damage'):\n    my_node.take_damage()"},
    {"q": "how to get a property by string name", "a": "var speed = my_node.get('movement_speed')"},
    {"q": "how to set a property by string name", "a": "my_node.set('movement_speed', 500)"},
    {"q": "how to yield to the next frame", "a": "await get_tree().process_frame"},
    {"q": "how to yield to the next physics frame", "a": "await get_tree().physics_frame"}
]

# Add the new examples to the main training list!
training_data.extend(new_advanced_data)

print(f"✅ 150+ NEW ADVANCED EXAMPLES ADDED!")
print(f"   Total Examples is now: {len(training_data)}")
print(f"   Run CELL 6 and CELL 7 again to train your AI on the new data!")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║         CELL 5.6 — 200 MORE ADVANCED EXAMPLES            ║
# ║  Multiplayer, Time, Quaternions, Callables, and more!    ║
# ╚══════════════════════════════════════════════════════════╝

even_more_data = [
    # --- MULTIPLAYER & NETWORKING ---
    {"q": "how to check if i am the multiplayer server", "a": "if multiplayer.is_server():\n    spawn_enemies()"},
    {"q": "how to get my unique multiplayer id", "a": "var my_id = multiplayer.get_unique_id()"},
    {"q": "how to get an array of all connected peer ids", "a": "var peers = multiplayer.get_peers()"},
    {"q": "how to create a multiplayer server", "a": "var peer = ENetMultiplayerPeer.new()\npeer.create_server(PORT)\nmultiplayer.multiplayer_peer = peer"},
    {"q": "how to join a multiplayer server as a client", "a": "var peer = ENetMultiplayerPeer.new()\npeer.create_client(IP_ADDRESS, PORT)\nmultiplayer.multiplayer_peer = peer"},
    {"q": "how to disconnect from multiplayer", "a": "multiplayer.multiplayer_peer = null"},
    {"q": "how to call an rpc function on all peers", "a": "rpc('my_remote_function', argument)"},
    {"q": "how to call an rpc function only on the server", "a": "rpc_id(1, 'my_server_function', argument)"},
    {"q": "how to call an rpc function on a specific peer", "a": "rpc_id(peer_id, 'send_private_message', text)"},
    {"q": "how to define an rpc function that anyone can call", "a": "@rpc('any_peer')\nfunc my_func():\n    pass"},
    {"q": "how to define an rpc function that runs locally and remotely", "a": "@rpc('call_local')\nfunc spawn_bullet():\n    pass"},
    {"q": "how to get the id of the peer who sent the rpc", "a": "var sender_id = multiplayer.get_remote_sender_id()"},

    # --- TIME AND DATES ---
    {"q": "how to get the current time in milliseconds", "a": "var ticks = Time.get_ticks_msec()"},
    {"q": "how to get the current time in microseconds", "a": "var ticks = Time.get_ticks_usec()"},
    {"q": "how to get the current system date and time as a dictionary", "a": "var datetime = Time.get_datetime_dict_from_system()"},
    {"q": "how to get the current year", "a": "var year = Time.get_datetime_dict_from_system()['year']"},
    {"q": "how to get the current unix timestamp", "a": "var unix_time = Time.get_unix_time_from_system()"},
    {"q": "how to get a time string from a unix timestamp", "a": "var time_str = Time.get_time_string_from_unix_time(unix_time)"},
    {"q": "how to get a date string from a unix timestamp", "a": "var date_str = Time.get_date_string_from_unix_time(unix_time)"},

    # --- CALLABLES AND LAMBDAS (GODOT 4) ---
    {"q": "how to create a callable from a function", "a": "var my_callable = Callable(self, 'my_function')"},
    {"q": "how to call a callable", "a": "my_callable.call()"},
    {"q": "how to call a callable with arguments", "a": "my_callable.call(arg1, arg2)"},
    {"q": "how to bind arguments to a callable", "a": "var bound_callable = my_callable.bind(extra_arg)"},
    {"q": "how to unbind arguments from a callable", "a": "var unbound = my_callable.unbind(1)"},
    {"q": "how to create an inline lambda function", "a": "var my_lambda = func(x): return x * 2"},
    {"q": "how to connect a signal to a lambda function", "a": "$Button.pressed.connect(func(): print('Clicked!'))"},
    {"q": "how to check if a callable is valid", "a": "if my_callable.is_valid():\n    my_callable.call()"},
    {"q": "how to check if a callable is null", "a": "if my_callable.is_null():\n    pass"},

    # --- VIEWPORT AND WINDOW ---
    {"q": "how to get the current viewport", "a": "var vp = get_viewport()"},
    {"q": "how to get the visible rectangle of the viewport", "a": "var rect = get_viewport_rect()"},
    {"q": "how to get the mouse position in the viewport", "a": "var mouse_pos = get_viewport().get_mouse_position()"},
    {"q": "how to get the main window", "a": "var window = get_window()"},
    {"q": "how to check if the window has focus", "a": "if get_window().has_focus():\n    pass"},
    {"q": "how to request attention for the window", "a": "get_window().request_attention()"},
    {"q": "how to set the window to always on top", "a": "get_window().always_on_top = true"},
    {"q": "how to make the window transparent", "a": "get_window().transparent = true"},
    {"q": "how to enable borderless window", "a": "get_window().borderless = true"},
    {"q": "how to get the safe area for mobile notches", "a": "var safe_rect = DisplayServer.get_display_safe_area()"},

    # --- 3D TRANSFORMS AND QUATERNIONS ---
    {"q": "how to get the global position in 3d", "a": "var pos = global_transform.origin"},
    {"q": "how to set the global position in 3d", "a": "global_transform.origin = Vector3(10, 0, 5)"},
    {"q": "how to translate a 3d object locally", "a": "translate_object_local(Vector3(0, 0, -1))"},
    {"q": "how to translate a 3d object globally", "a": "global_translate(Vector3(0, 1, 0))"},
    {"q": "how to scale a 3d object", "a": "scale = Vector3(2, 2, 2)"},
    {"q": "how to get the rotation quaternion", "a": "var quat = quaternion"},
    {"q": "how to create a quaternion from an axis and angle", "a": "var quat = Quaternion(Vector3.UP, deg_to_rad(45))"},
    {"q": "how to slerp between two quaternions", "a": "quaternion = quaternion.slerp(target_quat, 0.1)"},
    {"q": "how to get the euler angles from a quaternion", "a": "var euler = quat.get_euler()"},
    {"q": "how to interpolate a transform3d", "a": "global_transform = global_transform.interpolate_with(target_transform, 0.1)"},
    {"q": "how to make a transform look at a target", "a": "var new_transform = transform.looking_at(target_pos, Vector3.UP)"},

    # --- 2D TRANSFORMS AND GEOMETRY ---
    {"q": "how to get the global position in 2d", "a": "var pos = global_position"},
    {"q": "how to set the global position in 2d", "a": "global_position = Vector2(100, 200)"},
    {"q": "how to get the global rotation in 2d", "a": "var rot = global_rotation"},
    {"q": "how to move a 2d object locally", "a": "translate(Vector2(10, 0))"},
    {"q": "how to check if a point is inside a polygon", "a": "if Geometry2D.is_point_in_polygon(point, polygon_array):\n    pass"},
    {"q": "how to get the closest point on a line segment", "a": "var closest = Geometry2D.get_closest_point_to_segment(point, line_start, line_end)"},
    {"q": "how to triangulate a polygon", "a": "var triangles = Geometry2D.triangulate_polygon(polygon_array)"},
    {"q": "how to clip two polygons", "a": "var result = Geometry2D.clip_polygons(poly_a, poly_b)"},

    # --- ADVANCED UI AND THEMES ---
    {"q": "how to get the default theme", "a": "var theme = ThemeDB.get_default_theme()"},
    {"q": "how to override a color in a control node", "a": "$Label.add_theme_color_override('font_color', Color.RED)"},
    {"q": "how to override a font size in a control node", "a": "$Label.add_theme_font_size_override('font_size', 24)"},
    {"q": "how to override a stylebox in a control node", "a": "$Panel.add_theme_stylebox_override('panel', my_stylebox)"},
    {"q": "how to remove a theme override", "a": "$Label.remove_theme_color_override('font_color')"},
    {"q": "how to check if a control has a theme override", "a": "if $Label.has_theme_color_override('font_color'):\n    pass"},
    {"q": "how to force a scroll container to scroll to the bottom", "a": "$ScrollContainer.scroll_vertical = $ScrollContainer.get_v_scroll_bar().max_value"},
    {"q": "how to ensure a control is visible in a scroll container", "a": "$ScrollContainer.ensure_control_visible($MyButton)"},
    {"q": "how to accept a gui input event so it doesnt pass through", "a": "accept_event()"},
    {"q": "how to set the mouse filter to ignore", "a": "mouse_filter = Control.MOUSE_FILTER_IGNORE"},
    {"q": "how to set the mouse filter to pass", "a": "mouse_filter = Control.MOUSE_FILTER_PASS"},
    {"q": "how to set the mouse filter to stop", "a": "mouse_filter = Control.MOUSE_FILTER_STOP"},
    {"q": "how to show a tooltip on hover", "a": "tooltip_text = 'This is a helpful tip!'"},

    # --- AUDIO 2D AND 3D ---
    {"q": "how to play a 2d sound", "a": "$AudioStreamPlayer2D.play()"},
    {"q": "how to set the max distance for a 2d sound", "a": "$AudioStreamPlayer2D.max_distance = 500.0"},
    {"q": "how to play a 3d sound", "a": "$AudioStreamPlayer3D.play()"},
    {"q": "how to set the max distance for a 3d sound", "a": "$AudioStreamPlayer3D.max_distance = 50.0"},
    {"q": "how to change the attenuation model of a 3d sound", "a": "$AudioStreamPlayer3D.attenuation_model = AudioStreamPlayer3D.ATTENUATION_INVERSE_DISTANCE"},
    {"q": "how to check if a sound finished playing", "a": "await $AudioStreamPlayer.finished"},
    {"q": "how to seek to a specific time in an audio stream", "a": "$AudioStreamPlayer.seek(10.5)"},
    {"q": "how to get the current playback position of an audio stream", "a": "var pos = $AudioStreamPlayer.get_playback_position()"},

    # --- PARTICLES AND VISUALS ---
    {"q": "how to set cpu particles to emit once", "a": "$CPUParticles2D.one_shot = true"},
    {"q": "how to change the amount of cpu particles", "a": "$CPUParticles2D.amount = 100"},
    {"q": "how to change the color of cpu particles", "a": "$CPUParticles2D.color = Color.RED"},
    {"q": "how to make gpu particles trail", "a": "$GPUParticles2D.trail_enabled = true"},
    {"q": "how to clear emitted gpu particles", "a": "$GPUParticles2D.restart()"},
    {"q": "how to set a light2d energy", "a": "$PointLight2D.energy = 2.0"},
    {"q": "how to set a light2d color", "a": "$PointLight2D.color = Color.YELLOW"},
    {"q": "how to enable shadows on a light2d", "a": "$PointLight2D.shadow_enabled = true"},
    {"q": "how to set a directional light 3d energy", "a": "$DirectionalLight3D.light_energy = 1.5"},

    # --- MOBILE AND TOUCH ---
    {"q": "how to check if the device has a touchscreen", "a": "if DisplayServer.is_touchscreen_available():\n    pass"},
    {"q": "how to show the virtual keyboard on mobile", "a": "DisplayServer.virtual_keyboard_show('')"},
    {"q": "how to hide the virtual keyboard on mobile", "a": "DisplayServer.virtual_keyboard_hide()"},
    {"q": "how to get the height of the virtual keyboard", "a": "var kb_height = DisplayServer.virtual_keyboard_get_height()"},
    {"q": "how to handle a screen touch event", "a": "func _input(event):\n    if event is InputEventScreenTouch and event.pressed:\n        print('Touched at: ', event.position)"},
    {"q": "how to handle a screen drag event", "a": "func _input(event):\n    if event is InputEventScreenDrag:\n        print('Dragged by: ', event.relative)"},
    {"q": "how to get the index of the finger touching the screen", "a": "var finger_id = event.index"},

    # --- REGEX AND STRING FORMATTING ---
    {"q": "how to create a regex object", "a": "var regex = RegEx.new()"},
    {"q": "how to compile a regex pattern", "a": "regex.compile('\\\\d+')"},
    {"q": "how to search a string using regex", "a": "var result = regex.search('There are 10 apples')"},
    {"q": "how to get the matched string from a regex result", "a": "var match_str = result.get_string()"},
    {"q": "how to find all regex matches in a string", "a": "var results = regex.search_all('10 apples and 5 oranges')"},
    {"q": "how to replace text using regex", "a": "var new_text = regex.sub('10 apples', 'many')"},
    {"q": "how to format a string using a dictionary", "a": "var text = 'Name: {name}, Age: {age}'.format({'name': 'John', 'age': 30})"},
    {"q": "how to format a string using an array", "a": "var text = 'Score: %s / %s' % [current, max]"},

    # --- NOISE AND RANDOMNESS ---
    {"q": "how to create a fast noise lite object", "a": "var noise = FastNoiseLite.new()"},
    {"q": "how to set the noise type", "a": "noise.noise_type = FastNoiseLite.TYPE_SIMPLEX"},
    {"q": "how to set the noise seed", "a": "noise.seed = randi()"},
    {"q": "how to get a 2d noise value", "a": "var val = noise.get_noise_2d(x, y)"},
    {"q": "how to get a 3d noise value", "a": "var val = noise.get_noise_3d(x, y, z)"},
    {"q": "how to generate a noise image", "a": "var img = noise.get_image(512, 512)"},
    {"q": "how to pick a random weighted item", "a": "var item = ['common', 'rare'].pick_random() # Needs custom logic for weights"},

    # --- STATE MACHINES AND LOGIC ---
    {"q": "how to use a match statement like a switch", "a": "match state:\n    'idle':\n        play('idle')\n    'run':\n        play('run')"},
    {"q": "how to match multiple values in a match statement", "a": "match state:\n    'jump', 'fall':\n        play('air')"},
    {"q": "how to use a default case in a match statement", "a": "match state:\n    _:\n        print('Unknown state')"},
    {"q": "how to use a ternary operator for an if else statement", "a": "var speed = 100 if is_running else 50"},
    {"q": "how to assert a condition is true for debugging", "a": "assert(health > 0, 'Health must be positive!')"},
    {"q": "how to check the type of a variable", "a": "if typeof(my_var) == TYPE_INT:\n    pass"},
    {"q": "how to check if an object is a specific class", "a": "if my_node is CharacterBody2D:\n    pass"},
    {"q": "how to cast a variable to a specific class", "a": "var player = my_node as CharacterBody2D"},

    # --- FILEACCESS ADVANCED ---
    {"q": "how to read a file line by line", "a": "var file = FileAccess.open('user://data.txt', FileAccess.READ)\nwhile not file.eof_reached():\n    var line = file.get_line()"},
    {"q": "how to store a float in a binary file", "a": "file.store_float(3.14)"},
    {"q": "how to read a float from a binary file", "a": "var val = file.get_float()"},
    {"q": "how to store a variable in a file", "a": "file.store_var(my_dict)"},
    {"q": "how to read a variable from a file", "a": "var my_dict = file.get_var()"},
    {"q": "how to get the length of a file", "a": "var length = file.get_length()"},
    {"q": "how to get the current position in a file", "a": "var pos = file.get_position()"},
    {"q": "how to seek to a position in a file", "a": "file.seek(100)"},
    {"q": "how to seek to the end of a file", "a": "file.seek_end()"},

    # --- MISC GODOT 4 FEATURES ---
    {"q": "how to create a custom resource script", "a": "class_name MyStats extends Resource\n@export var health: int = 100"},
    {"q": "how to export a node path", "a": "@export var target_node: NodePath"},
    {"q": "how to export a node directly in godot 4", "a": "@export var target: Node2D"},
    {"q": "how to export an array of strings", "a": "@export var names: Array[String]"},
    {"q": "how to export an enum dropdown", "a": "enum State {IDLE, RUN}\n@export var current_state: State"},
    {"q": "how to mark a variable to save in the editor", "a": "@export var speed: float = 10.0"},
    {"q": "how to run a script in the editor", "a": "@tool\nextends Node"},
    {"q": "how to notify the editor to redraw", "a": "queue_redraw()"},
    {"q": "how to draw a line in canvasitem", "a": "func _draw():\n    draw_line(Vector2(0,0), Vector2(100,100), Color.RED, 2.0)"},
    {"q": "how to draw a circle in canvasitem", "a": "func _draw():\n    draw_circle(Vector2(50,50), 20.0, Color.BLUE)"},
    {"q": "how to draw a rectangle in canvasitem", "a": "func _draw():\n    draw_rect(Rect2(10, 10, 50, 50), Color.GREEN)"}
]

# Add the new examples to the main training list!
training_data.extend(even_more_data)

print(f"✅ 200+ MORE ADVANCED EXAMPLES ADDED!")
print(f"   Total Examples is now: {len(training_data)}")
print(f"   Run CELL 6 and CELL 7 again to train your AI on the new data!")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║         CELL 5.7 — MOBILE & DESKTOP/PC EXAMPLES          ║
# ║  Godot 4.x Cross-Platform, Sensors, Windows, and Touch   ║
# ╚══════════════════════════════════════════════════════════╝

platform_data = [
    # --- CROSS-PLATFORM CHECKS ---
    {"q": "how to check if the game is running on mobile", "a": "if OS.has_feature('mobile'):\n    setup_touch_controls()"},
    {"q": "how to check if the game is running on pc or desktop", "a": "if OS.has_feature('pc'):\n    setup_keyboard_controls()"},
    {"q": "how to check if the game is running on android", "a": "if OS.has_feature('android'):\n    pass"},
    {"q": "how to check if the game is running on windows", "a": "if OS.has_feature('windows'):\n    pass"},
    {"q": "how to check if the game is running on web html5", "a": "if OS.has_feature('web'):\n    pass"},
    {"q": "how to get the name of the operating system", "a": "var os_name = OS.get_name() # Returns 'Windows', 'Android', 'macOS', etc."},

    # --- MOBILE SPECIFIC (SENSORS & HAPTICS) ---
    {"q": "how to vibrate an android or ios phone", "a": "Input.vibrate_handheld(500) # Vibrates for 500 milliseconds"},
    {"q": "how to get the phone accelerometer data", "a": "var accel = Input.get_accelerometer() # Returns Vector3"},
    {"q": "how to get the phone gyroscope data", "a": "var gyro = Input.get_gyroscope() # Returns Vector3"},
    {"q": "how to get the phone magnetometer data", "a": "var mag = Input.get_magnetometer() # Returns Vector3"},
    {"q": "how to get the phone gravity sensor data", "a": "var gravity = Input.get_gravity() # Returns Vector3"},
    {"q": "how to check if the phone has a touch screen", "a": "if DisplayServer.is_touchscreen_available():\n    show_on_screen_buttons()"},
    {"q": "how to prevent the phone screen from sleeping", "a": "DisplayServer.window_set_keep_screen_on(true)"},

    # --- MOBILE SPECIFIC (TOUCH & GESTURES) ---
    {"q": "how to detect a multi touch pinch gesture", "a": "if event is InputEventScreenPinch:\n    camera.zoom *= event.scale"},
    {"q": "how to detect a multi touch drag or pan", "a": "if event is InputEventScreenDrag and event.index == 1:\n    print('Second finger dragging!')"},
    {"q": "how to get the position of a specific touch index", "a": "if event is InputEventScreenTouch and event.index == 0:\n    var pos = event.position"},
    {"q": "how to show the mobile virtual keyboard", "a": "DisplayServer.virtual_keyboard_show('')"},
    {"q": "how to hide the mobile virtual keyboard", "a": "DisplayServer.virtual_keyboard_hide()"},
    {"q": "how to get the height of the mobile keyboard to move ui up", "a": "var kb_height = DisplayServer.virtual_keyboard_get_height()"},
    {"q": "how to get the safe area for phones with notches", "a": "var safe_area = DisplayServer.get_display_safe_area()"},

    # --- DESKTOP/PC SPECIFIC (WINDOW MANAGEMENT) ---
    {"q": "how to maximize the window on pc", "a": "DisplayServer.window_set_mode(DisplayServer.WINDOW_MODE_MAXIMIZED)"},
    {"q": "how to minimize the window on pc", "a": "DisplayServer.window_set_mode(DisplayServer.WINDOW_MODE_MINIMIZED)"},
    {"q": "how to make the game fullscreen on pc", "a": "DisplayServer.window_set_mode(DisplayServer.WINDOW_MODE_FULLSCREEN)"},
    {"q": "how to make the game windowed on pc", "a": "DisplayServer.window_set_mode(DisplayServer.WINDOW_MODE_WINDOWED)"},
    {"q": "how to make a borderless window on pc", "a": "DisplayServer.window_set_flag(DisplayServer.WINDOW_FLAG_BORDERLESS, true)"},
    {"q": "how to make the window always on top", "a": "DisplayServer.window_set_flag(DisplayServer.WINDOW_FLAG_ALWAYS_ON_TOP, true)"},
    {"q": "how to make the window background transparent", "a": "get_viewport().transparent_bg = true\nDisplayServer.window_set_flag(DisplayServer.WINDOW_FLAG_TRANSPARENT, true)"},
    {"q": "how to disable window resizing by the user", "a": "DisplayServer.window_set_flag(DisplayServer.WINDOW_FLAG_RESIZE_DISABLED, true)"},
    {"q": "how to request the user's attention on the taskbar", "a": "DisplayServer.window_request_attention()"},

    # --- DESKTOP/PC SPECIFIC (MONITORS & MOUSE) ---
    {"q": "how to get the number of monitors connected", "a": "var monitor_count = DisplayServer.get_screen_count()"},
    {"q": "how to move the game window to the second monitor", "a": "DisplayServer.window_set_current_screen(1)"},
    {"q": "how to get the refresh rate of the current monitor", "a": "var hz = DisplayServer.screen_get_refresh_rate()"},
    {"q": "how to lock the mouse to the game window for a 3d game", "a": "Input.set_mouse_mode(Input.MOUSE_MODE_CAPTURED)"},
    {"q": "how to confine the mouse so it cannot leave the window", "a": "Input.set_mouse_mode(Input.MOUSE_MODE_CONFINED)"},
    {"q": "how to free the mouse", "a": "Input.set_mouse_mode(Input.MOUSE_MODE_VISIBLE)"},
    {"q": "how to change the mouse cursor to a custom image", "a": "Input.set_custom_mouse_cursor(preload('res://cursor.png'))"},
    
    # --- DESKTOP/PC SPECIFIC (FILES & OS) ---
    {"q": "how to open the user's default web browser to a url", "a": "OS.shell_open('https://godotengine.org')"},
    {"q": "how to open a file explorer folder on pc", "a": "OS.shell_open(ProjectSettings.globalize_path('user://'))"},
    {"q": "how to copy text to the pc clipboard", "a": "DisplayServer.clipboard_set('Text copied!')"},
    {"q": "how to paste text from the pc clipboard", "a": "var text = DisplayServer.clipboard_get()"},
    {"q": "how to execute a command line program on pc", "a": "var output = []\nOS.execute('ping', ['google.com'], output)"},
    {"q": "how to get the path to the user's documents folder", "a": "var docs = OS.get_system_dir(OS.SYSTEM_DIR_DOCUMENTS)"},
    {"q": "how to get the path to the user's downloads folder", "a": "var downloads = OS.get_system_dir(OS.SYSTEM_DIR_DOWNLOADS)"}
]

# Add the new examples to the main training list!
training_data.extend(platform_data)

print(f"✅ MOBILE & DESKTOP EXAMPLES ADDED!")
print(f"   Total Examples is now: {len(training_data)}")
print(f"   Run CELL 6 and CELL 7 again to train your AI on the new data!")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6: 100% CUSTOM GODOT TOKENIZER (ZERO DOWNLOADS)    ║
# ╚══════════════════════════════════════════════════════════╝
import torch
from torch.utils.data import Dataset, DataLoader
import random

# We import the raw math builders to make our own dictionary from scratch!
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

print("⏳ 1. Generating MASSIVE Godot Dataset...")
nodes = ["Sprite2D", "CharacterBody2D", "RigidBody2D", "Area2D", "Camera2D", "Label", "Button", "AudioStreamPlayer", "Timer", "RayCast2D", "AnimationPlayer"]
actions = ["hide", "show", "queue_free", "get_parent", "get_child", "duplicate", "set_process"]
properties = ["position", "rotation", "scale", "visible", "modulate", "name", "z_index"]

massive_data = []

# Generate 5,000 Action Examples
for _ in range(5000):
    node = random.choice(nodes)
    action = random.choice(actions)
    q = f"how to {action} a {node} in godot 4"
    a = f"var my_node = ${node}\nmy_node.{action}()"
    massive_data.append({"q": q, "a": a})

# Generate 5,000 Property Examples
for _ in range(5000):
    node = random.choice(nodes)
    prop = random.choice(properties)
    val = random.choice(["Vector2(0,0)", "true", "false", "Color(1,1,1)", "1.5", "'NewName'"])
    q = f"how to change the {prop} of {node}"
    a = f"${node}.{prop} = {val}"
    massive_data.append({"q": q, "a": a})

final_training_data = training_data + massive_data
print(f"✅ Massive Data Ready! Total Examples: {len(final_training_data)}")

# --- TRAIN A 100% CUSTOM TOKENIZER FROM SCRATCH ---
print("⏳ 2. Building a Custom Dictionary from scratch (NO DOWNLOADS!)...")

# Create an empty brain for reading words
raw_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
raw_tokenizer.pre_tokenizer = Whitespace()

# Tell it to learn a maximum of 5,000 Godot-specific words
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[EOS]", "User:", "Assistant:"], vocab_size=5000)

# Feed it ONLY your Godot data
def get_training_corpus():
    for item in final_training_data:
        yield f"User: {item['q']} Assistant: {item['a']}[EOS]"

# Train the dictionary!
raw_tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

# Wrap it so PyTorch can use it
tokenizer = PreTrainedTokenizerFast(tokenizer_object=raw_tokenizer)
tokenizer.pad_token = "[PAD]"
tokenizer.eos_token = "[EOS]"

print(f"✅ Custom Tokenizer Built! It knows exactly {len(tokenizer)} Godot words.")

# --- NEW DATASET CLASS ---
class ProGodotDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_length=256):
        self.input_ids = []
        for item in data_list:
            text = f"User: {item['q']} Assistant: {item['a']}{tokenizer.eos_token}"
            encoded = tokenizer(text, truncation=True, max_length=max_length, padding="max_length")
            self.input_ids.append(encoded['input_ids'])

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.input_ids[idx])

print("⏳ 3. Encoding 10,000+ examples into numbers...")
# We can safely go back to max_length=256 and batch_size=64!
dataset = ProGodotDataset(final_training_data, tokenizer, max_length=256)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# --- REBUILD THE AI BRAIN ---
print(f"🧠 Rebuilding AI with {len(tokenizer)} Vocabulary Size...")
your_ai = YourGameAI(
    vocab_size=len(tokenizer), # This is now only ~5,000!
    embed_dim=512,
    num_layers=6,
    num_heads=8,
    ff_dim=2048,
    max_length=256,
    dropout=0.2
).to(device)

print(f"✅ AI Rebuilt! Parameters: {sum(p.numel() for p in your_ai.parameters()):,}")
print("🚀 READY TO TRAIN! Go run CELL 7 again!")

In [ ]:
# CELL 7 — TRAIN YOUR AI
# Improved with repetition penalty to stop word loops
# Uses User/Assistant format in test prompt

import torch.optim as optim
import time

def train_your_ai(
    model,
    dataloader,
    tokenizer,
    num_epochs=100,
    learning_rate=5e-4,
    save_dir=SAVE
):
    # Optimizer adjusts the AI weights during training
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.05)

    # ADVANCED AI: OneCycleLR is the industry standard for Transformers
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=learning_rate, 
        steps_per_epoch=len(dataloader), 
        epochs=num_epochs,
        pct_start=0.1
    )

    # ADVANCED AI: Label smoothing prevents the AI from becoming overconfident
    loss_fn = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
    
    # ADVANCED AI: Mixed Precision Scaler (Trains 2x Faster on Kaggle GPU!)
    scaler = torch.amp.GradScaler(device)

    best_loss = float('inf')
    history = []

    print("🚀 TRAINING YOUR AI STARTED!")
    print(f"   Epochs: {num_epochs}")
    print(f"   Examples: {len(dataloader.dataset)}")
    print("=" * 50)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_batches = 0
        start_time = time.time()

        for batch in dataloader:
            batch = batch.to(device)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            optimizer.zero_grad()

            # ADVANCED AI: Autocast uses 16-bit math for lightning-fast training
            with torch.amp.autocast(device_type=device):
                logits = model(inputs)
                loss = loss_fn(
                    logits.reshape(-1, logits.size(-1)),
                    targets.reshape(-1)
                )

            # ADVANCED AI: Scaled backward pass
            scaler.scale(loss).backward()
            
            # Unscale before clipping gradients
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            
            # --- THE FIX FOR THE RED WARNING ---
            # Only step the scheduler if the optimizer actually took a step!
            scale_before = scaler.get_scale()
            scaler.update()
            scale_after = scaler.get_scale()
            
            if scale_before <= scale_after:
                scheduler.step()
            # -----------------------------------

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        elapsed = time.time() - start_time
        history.append(avg_loss)

        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {(epoch+1):>3}/{num_epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Time: {elapsed:.1f}s | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")

            # Test what AI says right now using NEW format
            model.eval()
            test_answer = model.generate(
                tokenizer,
                'User: how do i make a player jump Assistant:',
                max_new_tokens=50,
                temperature=0.9,
                repetition_penalty=1.3   # ← STOPS word loops!
            )
            print(f"   AI answer preview: {test_answer[:80]}...")

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'loss': best_loss,
                'vocab_size': len(tokenizer) # <--- FIXED for Hugging Face
            }, f'{save_dir}/checkpoints/best_model.pt')

        # Save checkpoint every 50 epochs
        if (epoch + 1) % 30 == 0:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'loss': avg_loss
            }, f'{save_dir}/checkpoints/epoch_{(epoch+1)}.pt')
            tokenizer.save_pretrained(f'{save_dir}/checkpoints') # <--- FIXED for Hugging Face
            print(f"💾 Saved checkpoint at epoch {epoch+1}")

    # Save final model
    torch.save(model.state_dict(), f'{save_dir}/model/final_model.pt')
    tokenizer.save_pretrained(f'{save_dir}/model') # <--- FIXED for Hugging Face

    print("\n" + "=" * 50)
    print("✅ TRAINING COMPLETE!")
    print(f"   Best loss: {best_loss:.4f}")
    print(f"   Final loss: {avg_loss:.4f}")
    print(f"   Lower loss = smarter AI")
    print(f"   Model saved to: {save_dir}/model/")
    return history
    
# START TRAINING YOUR AI
history = train_your_ai(
    model=your_ai,
    dataloader=dataloader,
    tokenizer=tokenizer,
    num_epochs=100,
    learning_rate=5e-4,
    save_dir=SAVE
)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║       STEP 4: CONTEXT MEMORY (CHATBOT MODE)              ║
# ╚══════════════════════════════════════════════════════════╝

# This string will hold the memory of your entire conversation
chat_memory = ""

def chat_with_godot_ai(model, tokenizer, new_question):
    global chat_memory
    model.eval()
    
    # 1. Add the new question to the memory
    chat_memory += f"User: {new_question} Assistant:"
    
    # 2. Convert the ENTIRE memory into numbers
    input_ids = tokenizer.encode(chat_memory, return_tensors="pt").to(device)
    
    # Prevent memory from getting too long and crashing the GPU
    if input_ids.shape[1] > 400:
        input_ids = input_ids[:, -400:] # Keep only the most recent conversation
    
    # 3. Generate the answer
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=150,
            temperature=0.3,
            repetition_penalty=1.2
        )
    
    # 4. Decode the answer
    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract ONLY the newest answer
    new_answer = full_output[len(chat_memory):].split("User:")[0].strip()
    
    # 5. Save the AI's answer into the memory so it remembers it for next time!
    chat_memory += f" {new_answer} \n"
    
    print(f"👤 YOU: {new_question}")
    print(f"🤖 AI:  {new_answer}")
    print("-" * 50)

# --- LET'S TEST ITS MEMORY! ---
print("Starting Chat... (Memory is empty)\n")

# Question 1
chat_with_godot_ai(your_ai, tokenizer, "how do i make a Sprite2D?")

# Question 2 (It has to remember what node we are talking about!)
chat_with_godot_ai(your_ai, tokenizer, "how do i hide it?")

# Question 3
chat_with_godot_ai(your_ai, tokenizer, "how do i delete it completely?")